In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
adata = ad.read_h5ad("./data/larry/larry_processed.h5ad")
adata_cospar = ad.read_h5ad("./data/larry/cospar_tmap_result.h5ad")
print(adata)
print(adata_cospar)

In [ ]:
import scvelo as scv

scv.tl.velocity_pseudotime(adata)
adata.obs["velocity_pseudotime"]

In [ ]:
import joblib
import numpy as np
from scripts.plotting import *

# load embedding
emb = joblib.load("./data/larry/larry_embedder.pkl")

X_emb = emb.X_emb        # shape (n_cells, 2)
pseudotime = adata.obs["velocity_pseudotime"].values

plot_velocity_streamplot(
    X_emb,
    tps_vf=emb.tps_vf,
    stream_density=0.9,
    streamline_thickness=5.0,
    arrowsize=1.5,
    scatter_size=8,
    scatter_alpha=0.6,
    cmap="coolwarm",
    show_axes=False,
    scatter_color=pseudotime
)

In [ ]:
def match_by_X_emb(adata, adata_cospar):
    X1 = adata.obsm["X_emb"]
    X2 = adata_cospar.obsm["X_emb"]

    rows1 = [tuple(r) for r in X1]
    rows2 = [tuple(r) for r in X2]

    lookup = {row: j for j, row in enumerate(rows2)}
    mapA2B = np.array([lookup.get(row, -1) for row in rows1], dtype=int)

    return mapA2B


mapA2B = match_by_X_emb(adata, adata_cospar)

print("Matched cells:", np.sum(mapA2B != -1), "/", len(mapA2B))

In [ ]:
# initialize with NaNs
adata_cospar.obs["velocity_pseudotime"] = np.nan

pt = adata.obs["velocity_pseudotime"].values

valid = mapA2B != -1
adata_cospar.obs.loc[
    adata_cospar.obs.index[mapA2B[valid]],
    "velocity_pseudotime"
] = pt[valid]

In [ ]:
genes = [
    "Npm1", "Set", "C1qbp", "Hspd1", "Hspa9", "Cd34",
    "Plac8", "Ctsc", "Cybb", "Srgn", "Prtn3", "Elane"
]

pt = adata_cospar.obs["velocity_pseudotime"].values
mask_pt = ~np.isnan(pt)

order = np.argsort(pt[mask_pt])

X = adata_cospar[mask_pt, genes].X

# force dense array (safe: cells × ~12 genes)
if not isinstance(X, np.ndarray):
    X = X.toarray()

X_sorted = X[order]

In [ ]:
X_z = (X_sorted - X_sorted.mean(axis=0)) / (X_sorted.std(axis=0) + 1e-8)
X_z = np.clip(X_z, -2, 2)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 3))

sns.heatmap(
    X_z.T,
    cmap="viridis",
    xticklabels=False,
    yticklabels=genes,
    cbar_kws={"label": "Z-scored expression"},
)

plt.xlabel("Velocity pseudotime →")
plt.ylabel("Gene")
plt.tight_layout()
plt.show()